<a target="_blank" href="https://colab.research.google.com/github/cboettig/rl-minicourse/blob/main/challenge.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

#  RL4Salmon Parameter Tests

In this example, we set up a generic three species, two action problem, and illustrate how to provide a custom population dynamics function, action function, and utility function to represent a caribou conservation objective.  

In [ ]:
# we'll need these packages to begin
!pip install stable-baselines3 plotnine polars sb3_contrib tensorboard


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.8/92.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 35.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu1

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1" # change to -1 if you want to use CPU

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
# pop = seals, lamprey, salmon
# Salmon Scenario
def dynamics(pop, effort, population_parameters, harvest_fn, p, timestep=1):

    pop = harvest_fn(pop, effort)
    X, Y, Z = pop[0], pop[1], pop[2]

    K_x = p["K_x"] + population_parameters[0]
    K_y = p["K_y"] + population_parameters[1]
    K_z = p["K_z"]
    v_x = p["v_x"]
    v_y = p["v_y"]
    h_x = p["h_x"]
    h_y = p["h_y"]
    D = p["D"]

    X += (p["r_x"] * X * (1 - X / K_x )
            - (Z * p["beta"] * (X**2)) / (v_x**2 + h_x * X**2 + h_y * Y**2)
            + p["sigma_x"] * X * np.random.normal()
            )

    Y += (p["r_y"] * Y * (1 - Y  / K_y )
            - (Z * p["beta"] * (Y**2)) / (v_y**2 + h_x * X**2 + h_y * Y**2)
            + p["sigma_y"] * Y * np.random.normal()
            )

    Z += p["alpha"] * p["beta"] * Z * (
            (X**2) / (v_x**2 + h_x * X**2 + h_y * Y**2)
            + (Y**2) / (v_y**2 + h_x * X**2 + h_y * Y**2)
            ) - p["dH"] * Z +  p["sigma_z"] * Z  * np.random.normal()



    pop = np.array([X, Y, Z], dtype=np.float32)
    pop = np.clip(pop, [0,0,0], [np.inf, np.inf, np.inf])
    return(pop)


Here, we change the sigma values to be negligible at 0.001 so that we can better visualize the underlying patterns in our environment. We also change the alpha value (seal growth rate) to be 0.05 to more accurately decrease the large timescale of the seal population.

In [ ]:
initial_pop = [0.5, 0.5, 0.2]


parameters = {
"r_x": np.float32(0.13),
"r_y": np.float32(0.2),
"r_z": np.float32(0.05),
"K_x": np.float32(0.5),
"K_y": np.float32(0.5),
"K_z": np.float32(1),
"h_x": np.float32(0.5),
"h_y": np.float32(0.5),
"v_x": np.float32(0.1),
"v_y": np.float32(1),
"beta": np.float32(.04),
"v0":  np.float32(0.1),
"D": np.float32(0.5),
"tau_yx": 0,
"tau_xy": 0,
"alpha": np.float32(.05),
"dH": np.float32(0.002),
"sigma_x": np.float32(0.001),
"sigma_y": np.float32(0.001),
"sigma_z": np.float32(0.001)
}


We must also define the dynamics of the action, a 'harvest' or culling function.  In this scenario, we imagine that we can cull either the salmon or pinniped population (or both).  We assume our control action introduces a percent mortality equal to the control effort applied times a catachability coefficient:

In [ ]:
def harvest(pop, effort):
    q0 = 0.5 # catchability coefficients -- erradication is impossible
    q2 = 0.5
    pop[0] = pop[0] * (1 - effort[0] * q0) # pop 0, seals
    pop[2] = pop[2] * (1 - effort[1] * q2) # pop 2, salmon
    return pop


Lastly, we need to define the utility or reward derived from taking these actions under this population state.  In this scenario, our population control actions are costly, while we acrue a benefit proportional to the size of the current salmon population:

In [ ]:
def utility(pop, effort):
    benefits = 0.5 * pop[1] # benefit from Salmon
    costs = .00001 * (effort[0] + effort[1]) # cost to culling
    if np.any(pop <= 0.01):
        benefits -= 1
    return benefits - costs




To simulate our environment and allow RL algorithms to train on this environment, we define a simple python class using the gym module.  This class defines the possible action space as two continuously-valued action variables (culling effort of salmon and pinnipeds respectively), and three continuously valued state variables (population of salmon, lamprey and pinnipeds).  To improve performance of RL training, it is necessary to transform the continuous space to -1, 1

In [ ]:
import gymnasium as gym

class s3a2(gym.Env):
    """A 3-species ecosystem model with two control actions"""
    def __init__(self, config=None):
        config = config or {}

        ## these parameters may be specified in config
        self.Tmax = config.get("Tmax", 10000)
        self.threshold = config.get("threshold", np.float32(1e-4))
        self.init_sigma = config.get("init_sigma", np.float32(1e-3))
        self.training = config.get("training", True)
        self.initial_pop = config.get("initial_pop", initial_pop)
        self.parameters = config.get("parameters", parameters)
        self.dynamics = config.get("dynamics", dynamics)
        self.harvest = config.get("harvest", harvest)
        self.utility = config.get("utility", utility)
        self.observe = config.get("observe", lambda state: state) # default to perfectly observed case
        self.bound = 2 * self.parameters["K_x"]

        self.action_space = gym.spaces.Box(
            np.array([-1, -1, 0, 0], dtype=np.float32),
            np.array([1, 1, 0.1, 0.1], dtype=np.float32),
            dtype = np.float32
        )
        self.observation_space = gym.spaces.Box(
            np.array([-1, -1, -1], dtype=np.float32),
            np.array([1, 1, 1], dtype=np.float32),
            dtype=np.float32,
        )
        self.reset(seed=config.get("seed", None))


    def reset(self, *, seed=None, options=None):
        self.timestep = 0
        self.initial_pop += np.multiply(self.initial_pop, np.float32(self.init_sigma * np.random.normal(size=3)))
        self.state = self.state_units(self.initial_pop)
        info = {}
        return self.observe(self.state), info


    def step(self, action):
        action = np.clip(action, self.action_space.low, self.action_space.high)
        pop = self.population_units(self.state) # current state in natural units
        human_actions = action[:2]
        pop_params = action[2:]
        effort = (human_actions + 1.) / 2

        # harvest and recruitment
        reward = self.utility(pop, effort)
        nextpop = self.dynamics(pop, effort, pop_params, self.harvest, self.parameters, self.timestep)

        self.timestep += 1
        terminated = bool(self.timestep > self.Tmax)

        # in training mode only: punish for population collapse
        if any(pop <= self.threshold) and self.training:
            terminated = True
            reward -= 50/self.timestep

        self.state = self.state_units(nextpop) # transform into [-1, 1] space
        observation = self.observe(self.state) # same as self.state
        return observation, reward, terminated, False, {}

    def state_units(self, pop):
        self.state = 2 * pop / self.bound - 1
        self.state = np.clip(self.state,
                             np.repeat(-1, self.state.__len__()),
                             np.repeat(1, self.state.__len__()))
        return np.float32(self.state)

    def population_units(self, state):
        pop = (state + 1) * self.bound /2
        return np.clip(pop,
                       np.repeat(0, pop.__len__()),
                       np.repeat(np.inf, pop.__len__()))

# verify that the environment is defined correctly
from stable_baselines3.common.env_checker import check_env
env = s3a2()
check_env(env, warn=True)


Here, we create a function to run a simulation of an episode by inputting an action, plotting the graph of the population proportions over time, and printing the population summary statistics including end population, mean, max, min, variance, and standard deviation. We also create a function to plot a phase plane of an episode which better visualizes the end behavior through limit cycles which are highlighted in yellow on the plot as yellow represents the end of the episode.

In [ ]:
from scipy.optimize import minimize
import polars as pl
from plotnine import ggplot, aes, geom_line, geom_path, labs

def get_last(col):
  return col.iloc[-1]

def get_func_pop(df, func):
  pd_df = df.to_pandas()
  func_pops = pd_df[["X", "Y", "Z"]].apply(func, axis=0)
  return func_pops

def obtain_simulated_data(action, timesteps=env.Tmax, env=env):
  df = []
  episode_reward = 0
  observation, _ = env.reset()
  for t in range(timesteps):
    obs = env.population_units(observation) # natural units
    df.append([t, episode_reward, *obs])
    observation, reward, terminated, done, info = env.step(action)
    episode_reward += reward

  df = pl.DataFrame(df, schema=["t", "reward", "X", "Y", "Z"])
  return df

def plot_simulation(df):
  cols = ["t", "reward", "X", "Y", "Z"]

  dfl = (pl.DataFrame(df, schema=cols).
          select(["t", "X", "Y", "Z"]).
          melt("t")
        )
  p = ggplot(dfl, aes("t", "value", color="variable")) + geom_path() + labs(title = "Environment Change Simulation")
  display(p)

def plot_phase_plane(df, X = "X", Y = "Y"):
  cols = ["t", "reward", "X", "Y", "Z"]

  dfl = (pl.DataFrame(df, schema=cols).
          select(["t", "X", "Y", "Z"])
        )
  p = ggplot(dfl, aes(X, Y, color="t")) + geom_path() + labs(title = "Environment Phase Plane Plot")
  display(p)


def run_simulation(action, timesteps=env.Tmax, env=env, focus=("X", "Y"), plot_funcs=(True, True)):
  df = obtain_simulated_data(action, timesteps, env)
  plot_simulation(df) if plot_funcs[0] else None
  X, Y = focus
  plot_phase_plane(df, X, Y) if plot_funcs[1] else None
  func_list = [get_last, np.mean, np.max, np.min, np.var, np.std]
  for func in func_list:
    display_final_pop(df, func)

def display_final_pop(df, func):
  info_dict = {get_last: "Final Population", np.mean: "Mean Population", np.max: "Max Population",
               np.min: "Min Population", np.var: "Population Variance", np.std: "Population SD"}
  func_pops = get_func_pop(df, func)
  func_suffix = info_dict[func]

  salmon_str = f"Salmon {func_suffix}: {func_pops[0]:.2f}"
  lamprey_str = f"Lamprey {func_suffix}: {func_pops[1]:.2f}"
  pinniped_str = f"Pinniped {func_suffix}: {func_pops[2]:.2f}"
  print(salmon_str + "\n" + lamprey_str + "\n" + pinniped_str + "\n")

zero_action = [-1, -1, 0, 0]

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def simulate_param_change_episode(h_x, h_y, v_x, v_y, beta, alpha, dH):
  curr_parameters = parameters.copy()
  curr_parameters["h_x"] = h_x
  curr_parameters["h_y"] = h_y
  curr_parameters["v_x"] = v_x
  curr_parameters["v_y"] = v_y
  curr_parameters["beta"] = beta
  curr_parameters["alpha"] = alpha
  curr_parameters["dH"] = dH
  curr_env = s3a2(config={"parameters": curr_parameters})
  run_simulation(zero_action, 10000, curr_env, plot_funcs=(True, False))

interact(simulate_param_change_episode,
         h_x=widgets.FloatSlider(min=0.1, max=1, step=0.1, value=0.500),
         h_y=widgets.FloatSlider(min=0.1, max=1, step=0.1, value=0.500),
         v_x=widgets.FloatSlider(min=0.1, max=1, step=0.1, value=0.500),
         v_y=widgets.FloatSlider(min=0.1, max=1, step=0.1, value=0.500),
         beta=widgets.FloatSlider(min=0.01, max=0.1, step=0.01, value=0.030),
         alpha=widgets.FloatSlider(min=0.01, max=1, step=0.05, value=0.060),
         dH=widgets.FloatSlider(min=0.001, max=0.003, step=0.0002, value=0.002))

interactive(children=(FloatSlider(value=0.5, description='h_x', max=1.0, min=0.1), FloatSlider(value=0.5, desc…

<function __main__.simulate_param_change_episode(h_x, h_y, v_x, v_y, beta, alpha, dH)>